# Lab 04: Regularized Regression and Stochastic Gradient Descent

In this lab, you will predict ocean temperature with Ridge, Lasso, and unregularized linear regression. You will then implement stochastic gradient descent (SGD) for linear regression and compare its coefficients with scikit-learn's ordinary least-squares estimates.

**Learning outcomes**

- Build a preprocessing pipeline without data leakage.
- Select regularization strength with cross-validation and interpret coefficient behavior.
- Compare regularized and unregularized predictive models.
- Implement and validate a stochastic-gradient-descent algorithm.

In [1]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV, LinearRegression, RidgeCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

## Data

This lab uses a course-selected subset of the [CalCOFI oceanographic bottle data](https://calcofi.org/data/oceanographic-data/bottle-database/). The file is hosted in the course data repository.

In [2]:
DATA_URL = (
    'https://raw.githubusercontent.com/rhodes-byu/stat-486/main/'
    'data/OceanicFisheries/ocean_data.csv'
)

df = pd.read_csv(DATA_URL)
df.head()

,T_degC,Salnty,O2ml_L,Depthm,Bottom_D,Wind_Spd,Dry_T,Wet_T,Wea,Cloud_Typ,Cloud_Amt,Visibility
0,16.830,33.8510,5.560,65,1337.0,14.0,16.5,15.5,NaN,NaN,NaN,NaN
1,9.262,33.8481,2.729,140,1202.0,5.0,15.0,13.0,Partly Cloudy,Stratus,1/10 or less but not zero,10km to 20km
2,15.390,33.4260,5.990,30,3871.0,10.0,18.8,17.6,Continuous blowing snow,Stratus,10/10,10km to 20km
3,14.540,32.9470,5.840,42,4018.0,14.0,16.9,16.1,Continuous blowing snow,Stratocumulus,10/10,4km to 10km
4,7.410,34.1810,1.000,300,4058.0,21.0,16.3,14.9,Partly Cloudy,Stratocumulus,7/10 to 8/10,4km to 10km


## Part 1: Regularized regression

The goal is to predict water temperature, `T_degC`. Use only numerical features; exclude `Wea`, `Cloud_Typ`, `Cloud_Amt`, and `Visibility`.

### 1. Prepare the data

1. Create training and test sets with `train_test_split`, using `random_state=307` and the default test size (25%).
2. Create a preprocessing pipeline that uses:
   - `SimpleImputer(strategy='mean')`,
   - `PolynomialFeatures(degree=2, include_bias=False)`, and
   - `StandardScaler`.
3. Fit the pipeline using only `X_train`, then transform both `X_train` and `X_test`.

In [3]:
# Your code here: data selection, the split, and preprocessing.
# Keep the unprocessed X_train and X_test available until after the split.
# Then use the fitted preprocessing pipeline to create processed versions.

X = df.drop(columns = ["T_degC", "Wea", "Cloud_Typ", "Cloud_Amt", "Visibility"])
y = df["T_degC"]

Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.25, random_state=307)

preprocessing = Pipeline([
    ("imputer", SimpleImputer(strategy='mean')),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler())
])

Xtrain_p = preprocessing.fit_transform(Xtrain)
Xtest_p = preprocessing.transform(Xtest)

### 2. Ridge regression

Fit `RidgeCV` with five-fold cross-validation and `np.logspace(-6, 6, 30)` as the candidate alpha values. Use the processed training data. Compute the test MSE with the processed test data.

In [4]:
# Your code here: fit the RidgeCV model, make test predictions, and calculate the requested quantities.
ridge = RidgeCV(alphas = np.logspace(-6, 6, 30), cv = 5)
ridge.fit(Xtrain_p, ytrain)

y_ridge = ridge.predict(Xtest_p)
r_mse = mean_squared_error(ytest, y_ridge)

print(ridge.alpha_)
print(r_mse)
print(ridge.coef_[np.argmax(np.abs(ridge.coef_))])
print(np.sum(ridge.coef_ == 0))

1e-06
1.7543290443387167
147.5640502282242
0


Write your response here.

**Your response for 2(b-e)**

- Optimal alpha: 1e-06
- Test MSE: 1.7543290443387167
- Largest-magnitude coefficient, excluding the intercept: 147.5640502282242
- Number of coefficients exactly equal to 0: 0

### 3. Lasso regression

Fit `LassoCV` with five-fold cross-validation and its default alpha search. Use the processed training data. Compute the test MSE with the processed test data.

In [5]:
# Your code here: fit the LassoCV model, make test predictions, and calculate the requested quantities.
lasso = LassoCV(cv = 5)
lasso.fit(Xtrain_p, ytrain)

y_lasso = lasso.predict(Xtest_p)
l_mse = mean_squared_error(ytest, y_lasso)

print(lasso.alpha_)
print(l_mse)
print(lasso.coef_[np.argmax(np.abs(lasso.coef_))])
print(np.sum(lasso.coef_ == 0))

0.011362725343010525
1.9812922233168826
2.447689386130063
19


c:\Users\JDele\Stat486\lab-04-jdleon4\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.402116e+00, tolerance: 5.899e+00
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\JDele\Stat486\lab-04-jdleon4\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.691929e+00, tolerance: 5.864e+00
  model = cd_fast.enet_coordinate_descent_gram(


Write your response here.

**Your response for 3(b-e)**

- Optimal alpha: 0.011362725343010525
- Test MSE: 1.9812922233168826
- Largest-magnitude coefficient, excluding the intercept: 2.447689386130063
- Number of coefficients exactly equal to 0: 19

### 4. Compare the models

1. Fit an unregularized `LinearRegression` model on the processed training data and report its test MSE.
2. Based on test MSE and the coefficient behavior you observed, recommend Lasso, Ridge, or unregularized regression for this problem. Justify your recommendation with specific evidence.

In [6]:
# Your code here: fit the unregularized model and calculate its test MSE.
lr = LinearRegression()
lr.fit(Xtrain_p, ytrain)

y_lr = lr.predict(Xtest_p)
lr_mse = mean_squared_error(ytest, y_lr)

print(lr_mse)

1.7543308610948378


Write your response here.

**Your response for 4(a-b)**

- Unregularized test MSE: 1.7543308610948378
- Recommendation and justification: I would probably use the Linear Regression method because the Ridge barely provides any improvement upon the MSE and the alpha states that feature selection made basically insignificant improvement to the predictions, thus, for simplicity, just use the Linear Regression

## Part 2: Stochastic gradient descent

### 5. Create an SGD function

Complete `sgd` in `sgd_function.py`. It must accept an `X` matrix, a target vector `y`, a learning rate (`alpha`), number of epochs, and a learning-rate decay parameter. It must return estimated regression coefficients.

The starter file includes `data_prep`, which adds an intercept column and prepares arrays for matrix operations. Follow the scaffold comments in `sgd_function.py`. In particular, each update must use one observation, every observation must be visited once per epoch in a random order, and the learning rate must decay after each epoch.

**Concept check:** In one or two sentences, explain how an SGD update differs from the batch-gradient update used in class.

SGD differs from batch-gradient descent because batch-gradient descent looks at the entire dataset before updating the betas, so it updates the betas once per epoch. SGD looks at each individual observation and updates the betas after each observation, meaning it updates the betas n times per epoch.

**Your response for the Part 2 concept check**



### 6. Test your function

1. Create a new preprocessing pipeline that imputes missing values and scales the features, but **does not** create polynomial features.
2. Apply this pipeline to all observations. A train/test split is not needed here because you are comparing estimated parameters.
3. Fit `LinearRegression` and report its intercept and `Salnty` coefficient.
4. In the code cell immediately below, import the completed `sgd` function from `sgd_function.py`. Your import must run successfully before you test the function.
5. Experiment with alpha and epochs until your SGD intercept and `Salnty` coefficient agree with the ordinary least-squares estimates to two decimal places. Set `random_state` when calling your function so your comparison is reproducible.

In [7]:
# Your code here: build the non-polynomial preprocessing pipeline, fit LinearRegression,
# and identify the intercept and Salnty coefficient.

new_processing = Pipeline([
    ("imputer", SimpleImputer(strategy = "mean")),
    ("scaler", StandardScaler())
])

Xnew_processed = new_processing.fit_transform(X)

lr = LinearRegression()
lr.fit(Xnew_processed, y)

print("Intercept:", lr.intercept_)
print("Salnty coefficient:", lr.coef_[X.columns.get_loc("Salnty")])

Intercept: 11.628595693779905
Salnty coefficient: 0.08672519432184468


In [8]:
# 6(d) Your code here:
# Import the completed `sgd` function from `sgd_function.py`.
# Hint: use the form `from module_name import function_name`.
# Verify that `sgd` is now a callable name before continuing.
from sgd_function import sgd

Write your response here.

**Your response for 6(c)**

- LinearRegression intercept: 11.628595693779905
- LinearRegression coefficient for `Salnty`: 0.08672519432184468

In [9]:
# Run the completed Part 6(d) import cell before continuing.
# Your code here: call your completed sgd function. Compare its intercept and Salnty coefficient
# with the LinearRegression estimates, adjusting alpha and epochs as needed.
betas = sgd(Xnew_processed, y, alpha=0.09, epochs=850, decay=0.99, random_state=307)

print("SGD intercept:", betas[0])
print("SGD Salnty coefficient:", betas[1])

SGD intercept: [11.62898925]
SGD Salnty coefficient: [0.08657259]


Write your response here.

**Your response for 6(e)**

- Alpha that worked: 0.09
- Epochs that worked: 850
- Brief comparison of your SGD and LinearRegression estimates: The SGD and LinearRegression estimates are very similar. The intercept is 11.6290 for SGD compared to 11.6286 for LinearRegression, while the Salnty coefficient is 0.08657 for SGD compared to 0.08673 for LinearRegression. Both estimates differ by less than 0.001, showing that the SGD model closely approximates the LinearRegression estimates.


## Before you submit

- Run all notebook cells from top to bottom.
- Replace every response placeholder.
- Confirm that `sgd_function.py` contains your completed function and your Part 6(d) import cell runs successfully.
- Run `uv sync` and commit the generated `uv.lock`.
- Commit and push the notebook, `sgd_function.py`, `pyproject.toml`, and `uv.lock` before submitting your repository URL to Canvas.